In [214]:
import pywencai
import pandas as pd
pd.set_option('display.max_columns', None)


# query = '扣非EPS 扣非净利润同比增长率 每股分红 每股净资产 销售净利率 机构持股占流通股比例 北向资金持股比例 最终控制人 商誉占净资产比例'
query = 'PE 扣非PE PB 扣非ROE 扣非净利润同比增长率 每股分红 分红比例 销售净利率  销售毛利率 商誉占净资产比例 ROA  ROE EPS 股息率  总营收增长率 资产负债率'
loop = True
query_type = 'stock'


r = pywencai.get(query=query,loop = loop, log = False, query_type = query_type)
r.columns = [x.split('[')[0] for x in r.columns.tolist()]



In [217]:
result = r[['股票代码','股票简称','市盈率(pe)','市盈率(pe,扣非ttm)','市净率(pb)','净资产收益率roe-扣除非经常损益','净资产收益率roe(加权,公布值)','总资产报酬率roa','股息率(股票获利率)','分红比例','销售毛利率','销售净利率','归属母公司股东的净利润-扣除非经常损益(同比增长率)','营业总收入(同比增长率)','商誉占净资产比例','资产负债率']]
result.columns = ['股票代码','股票简称','PE','扣非PE','PB','扣非ROE','ROE','ROA','股息率','分红比例','毛利率','净利率','扣非净利润增速','营收增速','商誉占净资产比例','资产负债率']

result = result.apply(pd.to_numeric, errors='ignore')

result['扣非PEG']  = (result['扣非PE'] / result['营收增速'])

def dividend_correct(x):
    if x >= 0.5:
        n = 1
    elif x < 0.25:
        n = 2
    else:
        n = 0.5/x
    return n

# def drop_duplicates(x):
#     x = ','.join(set(x.split('||')))
#     return x
# result['最终控制人类型'] = result['最终控制人类型'].apply(drop_duplicates)
# result['最终控制人'] = result['最终控制人'].apply(drop_duplicates)
result['分红比例'] = result['分红比例'].fillna(0)
result['扣非市赚率'] = result['扣非PE'] / result['扣非ROE'] * result['分红比例'].apply(dividend_correct)
result['市赚率'] =result['PE'] / result['ROE'] * result['分红比例'].apply(dividend_correct)


df = result.fillna('').apply(pd.to_numeric, errors='ignore').round(2).copy()

In [218]:
# 市赚率
# PE<0 ROE > 0, 亏损
# 扣非PE<0 扣非ROE>0, 扣非亏损

# ROE<0 资不抵债

df.loc[(df['PE']< 0) & (df['ROE']<0),'市赚率'] = '亏损'
df.loc[(df['PE']> 0) & (df['ROE']<0),'市赚率'] = '转盈'
df.loc[(df['PE']< 0) & (df['ROE']>0),'市赚率'] = '转亏'

df.loc[(df['扣非PE']< 0) & (df['扣非ROE']<0),'扣非市赚率'] = '扣非亏损'
df.loc[(df['扣非PE']> 0) & (df['扣非ROE']<0),'扣非市赚率'] = '转盈'
df.loc[(df['扣非PE']< 0) & (df['扣非ROE']>0),'扣非市赚率'] = '扣非转亏'
df.loc[(df['扣非PE']> 0) & (df['扣非ROE']<0) & (df['PE']< 0),'扣非市赚率'] = '扣非亏损'

df.loc[df['扣非PE']< 0,'扣非PEG'] = '扣非亏损'
# 分红比例 < 0

# PE 或 盈利增速 < 0

In [219]:
df.fillna('')

,股票代码,股票简称,PE,扣非PE,PB,扣非ROE,ROE,ROA,股息率,分红比例,毛利率,净利率,扣非净利润增速,营收增速,商誉占净资产比例,资产负债率,扣非PEG,扣非市赚率,市赚率
0,603013.SH,亚普股份,15.89,15.11,1.96,13.7,12.16,9.05,2.31,43.99,15.61,5.88,5.49,1.66,,35.23,9.12,1.1,1.31
1,837403.BJ,康农种业,18.54,17.36,1.77,19.06,21.47,11.58,1.37,51.29,30.73,18.62,31.65,45.85,,49.37,0.38,0.91,0.86
2,000538.SZ,云南白药,14.68,24.66,2.42,9.6,10.51,8.5,4.23,90.53,26.51,10.54,16.45,7.19,0.26,25.8,3.43,2.57,1.4
3,002763.SZ,汇洁股份,8.07,18.8,1.46,8.52,9.45,10.77,4.76,90.15,67.74,7.21,39.2,19.3,0.57,24.84,0.97,2.21,0.85
4,836871.BJ,派特尔,20.63,22.22,1.88,8.67,9.67,9.36,2.27,64.17,27.79,14.64,22.52,8.39,,9.6,2.65,2.56,2.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5330,601068.SH,中铝国际,33.87,-4.8,5.96,-41.39,-41.87,-6.03,,0.00,8.76,-12.69,-607.25,-5.74,0.01,82.28,扣非亏损,扣非亏损,转盈
5331,002650.SZ,加加食品,125.88,-16.48,1.57,-9.1,-8.69,-6.99,,0.00,18.71,-13.15,-114.93,-13.78,,21.31,扣非亏损,扣非亏损,转盈
5332,600117.SH,*ST西钢,-11.36,-5.5,1.62,-70.03,992.07,10.35,,0.00,-6.66,25.36,-39.38,-36.33,2.0,45.07,扣非亏损,扣非亏损,转亏
5333,300249.SZ,依米康,156.65,-23.37,7.92,-32.02,-40.77,-12.73,,0.00,14.5,-30.36,-33.16,-8.84,3.66,74.32,扣非亏损,扣非亏损,转盈


In [149]:
result

,股票代码,股票简称,PE,扣非PE,PB,扣非ROE,ROE,ROA,股息率,分红比例,毛利率,净利率,扣非净利润增速,营收增速,商誉占净资产比例,资产负债率,扣非PEG,扣非市赚率,市赚率
0,837403.BJ,康农种业,13.91,17.02,2.72,19.06,21.47,11.58,1.37,51.29,30.73,18.62,31.65,45.85,0.00,49.37,0.01,0.89,0.73
1,000049.SZ,德赛电池,47.12,16.90,1.29,10.33,12.72,4.17,1.66,30.81,9.13,2.52,-34.33,-6.73,0.00,61.35,-0.00,1.64,4.56
2,002402.SZ,和而泰,28.58,36.30,2.38,6.76,7.53,4.22,1.05,42.10,19.58,4.63,-21.85,25.85,10.32,46.98,0.05,5.37,4.23
3,300865.SZ,大宏立,-354.19,-31.43,1.53,-4.14,-3.77,-3.59,0.24,-15.85,17.76,-6.10,-386.37,9.40,0.00,25.68,0.00,15.18,171.11
4,301153.SZ,中科江南,-203.69,41.86,6.12,17.22,18.21,12.42,1.28,116.80,56.40,25.18,19.49,32.31,0.06,23.93,0.04,2.43,-11.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5359,873806.BJ,云星宇,22.01,31.81,2.43,7.68,11.45,3.68,0.00,26.17,14.05,5.26,22.36,-8.14,0.25,66.78,-0.00,4.14,2.87
5360,873833.BJ,美心翼申,17.87,30.05,1.23,5.04,8.65,7.15,1.99,56.79,23.11,9.56,-44.53,-11.52,0.00,18.76,-0.00,5.96,3.54
5361,603361.SH,浙江国祥,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5362,688530.SH,欧莱新材,0.00,0.00,0.00,7.67,10.20,7.65,0.00,0.00,21.99,10.36,53.65,21.50,0.00,36.15,0.00,0.00,0.00


In [220]:
df.to_csv('values.csv', index =False)